In [1]:
import cv2
from ultralytics import YOLO
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime
from IPython.display import display, clear_output
import PIL.Image


In [2]:
model = YOLO("../MODELS/yolov8n.pt")     # Adjust path if needed
cap = cv2.VideoCapture("../DATA/input_video.mp4")

print("Model Loaded ✔")
print("Video Loaded ✔")


Model Loaded ✔
Video Loaded ✔


In [3]:
os.makedirs("../OUTPUT/violations", exist_ok=True)

log_path = "../OUTPUT/violations_log.csv"

if not os.path.exists(log_path):
    df = pd.DataFrame(columns=["time", "violation_type", "image_path"])
    df.to_csv(log_path, index=False)


In [4]:
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH

    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    return interArea / float(boxAArea + boxBArea - interArea)


In [5]:
def detect_helmet_violations(detections, iou_thresh=0.2):
    bikes = [d for d in detections if d["class"] in ["motorbike", "bike"]]
    persons = [d for d in detections if d["class"] == "person"]
    helmets = [d for d in detections if d["class"] in ["helmet", "with_helmet"]]

    violations = []

    for bike in bikes:
        rider = None

        # match rider with bike
        for p in persons:
            if iou(bike["bbox"], p["bbox"]) > 0.2:
                rider = p
                break

        if rider is None:
            continue

        rx1, ry1, rx2, ry2 = rider["bbox"]
        head_zone = (rx1, ry1, rx2, ry1 + int(0.4 * (ry2 - ry1)))

        has_helmet = False
        for h in helmets:
            if iou(head_zone, h["bbox"]) > iou_thresh:
                has_helmet = True
                break

        if not has_helmet:
            violations.append({
                "type": "helmet_violation",
                "bike": bike["bbox"],
                "rider": rider["bbox"]
            })

    return violations


In [6]:
STOP_LINE_Y = 350     # adjust this after seeing video
TRAFFIC_LIGHT = "RED" # can automate later

def detect_signal_violations(detections, stop_line_y, traffic_light):
    if traffic_light != "RED":
        return []

    vehicles = [
        d for d in detections 
        if d["class"] in ["car", "motorbike", "bike", "truck", "bus"]
    ]

    violations = []

    for v in vehicles:
        x1, y1, x2, y2 = v["bbox"]
        vehicle_front_y = y2

        if vehicle_front_y < stop_line_y:
            violations.append({
                "type": "signal_violation",
                "bbox": (x1, y1, x2, y2)
            })

    return violations


In [7]:
def save_violation(violation_type, frame):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S%f")
    filename = f"violation_{violation_type}_{timestamp}.jpg"
    filepath = f"../OUTPUT/violations/{filename}"

    cv2.imwrite(filepath, frame)

    df = pd.read_csv("../OUTPUT/violations_log.csv")
    df.loc[len(df)] = [timestamp, violation_type, filepath]
    df.to_csv("../OUTPUT/violations_log.csv", index=False)

    return filepath


In [8]:
fps_delay = 0.05

while True:
    ret, frame = cap.read()
    if not ret:
        print("Video ended.")
        break

    results = model(frame, verbose=False)
    annotated_frame = results[0].plot()

    detections = []

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = int(box.cls[0])
        cls_name = model.names[cls]

        detections.append({
            "class": cls_name,
            "bbox": (x1, y1, x2, y2)
        })

    # ---------------- HELMET VIOLATIONS ----------------
    helmet_violations = detect_helmet_violations(detections)

    for v in helmet_violations:
        bx1, by1, bx2, by2 = v["bike"]
        cv2.rectangle(annotated_frame, (bx1, by1), (bx2, by2), (0,0,255), 3)
        cv2.putText(annotated_frame, "NO HELMET", (bx1, by1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,0,0), 2)
        save_violation("helmet", annotated_frame)

    # ---------------- SIGNAL VIOLATIONS ----------------
    signal_violations = detect_signal_violations(detections, STOP_LINE_Y, TRAFFIC_LIGHT)

    for v in signal_violations:
        x1, y1, x2, y2 = v["bbox"]
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0,255,255), 3)
        cv2.putText(annotated_frame, "SIGNAL VIOLATION", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
        save_violation("signal", annotated_frame)

    # ---------------- DRAW STOP LINE -------------------
    cv2.line(annotated_frame, (0, STOP_LINE_Y),
             (annotated_frame.shape[1], STOP_LINE_Y),
             (255,0,0), 2)

    cv2.putText(annotated_frame, f"Signal: {TRAFFIC_LIGHT}", (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    # ---------------- DISPLAY FRAME ---------------------
    clear_output(wait=True)
    rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
    display(PIL.Image.fromarray(rgb))

    time.sleep(fps_delay)

cap.release()
print("Done.")


KeyboardInterrupt: 